# 01 · SpaceX launch data collection — API

**Project context and attribution.** Developed for the IBM Applied Data Science Capstone (Coursera / IBM Skills Network). IBM supplied the original educational project framework and public teaching datasets; this is a reorganized, independently worded notebook incorporating the student's analysis and subsequent code corrections, with AI-assisted review. The archived teaching data are historical and are **not** a current account of SpaceX launches.

**Run environment:** Python 3 in VS Code's Jupyter notebook extension. Install the project dependencies outside the notebook using the repository's `requirements.txt`. There are no `pip`, `piplite`, JavaScript `fetch`, or notebook-environment-specific installation cells here.

## Aim and method

Read the **fixed historical SpaceX launch snapshot** used by the IBM capstone, and attempt to enrich its rocket, payload, launchpad, and core identifiers through the public SpaceX REST API. The analysis covers the original teaching period (up to **13 November 2020**) and focuses on single-core, single-payload launches.

**Service availability and reproducibility:** The historical SpaceX API has been observed returning HTTP 525. If any resource lookup fails, this notebook automatically uses IBM Skills Network's already-enriched, archived `dataset_part_1.csv`. This is a transparent fallback, **not** a successful live API request. The final CSV has the same schema either way, and the chosen source is printed alongside its output.

In [1]:
from pathlib import Path
from functools import lru_cache
from io import BytesIO
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPACE_X_API = "https://api.spacexdata.com/v4"
IBM_DATASETS = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets"
STATIC_LAUNCHES_URL = f"{IBM_DATASETS}/API_call_spacex_api.json"
ARCHIVED_FALCON9_URL = f"{IBM_DATASETS}/dataset_part_1.csv"
CUTOFF = pd.Timestamp("2020-11-13", tz="UTC")

session = requests.Session()
session.headers.update({"User-Agent": "SpaceX-DataScience-Capstone/1.0 (educational analysis)"})

## Retrieve the archived launch list

Use `raise_for_status()` and an explicit timeout so inaccessible sources produce a clear error instead of silently returning a broken dataset.

In [2]:
response = session.get(STATIC_LAUNCHES_URL, timeout=35)
response.raise_for_status()
launches = pd.json_normalize(response.json())
print("Archived launch records:", len(launches))
launches[["flight_number", "rocket", "date_utc"]].head()

Archived launch records: 107


,flight_number,rocket,date_utc
0,1,5e9d0d95eda69955f709d1eb,2006-03-24T22:30:00.000Z
1,2,5e9d0d95eda69955f709d1eb,2007-03-21T01:10:00.000Z
2,3,5e9d0d95eda69955f709d1eb,2008-08-03T03:34:00.000Z
3,4,5e9d0d95eda69955f709d1eb,2008-09-28T23:15:00.000Z
4,5,5e9d0d95eda69955f709d1eb,2009-07-13T03:35:00.000Z


## Restrict the observation period and record shape

The original project focuses on single-core launches with a single payload. Convert the dates before filtering and copy the subset to avoid chained-assignment warnings.

In [3]:
required = ["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]
missing = sorted(set(required) - set(launches.columns))
if missing:
    raise ValueError(f"Launch snapshot is missing columns: {missing}")

launch_subset = launches.loc[:, required].copy()
valid = (
    launch_subset["cores"].apply(lambda x: isinstance(x, list) and len(x) == 1)
    & launch_subset["payloads"].apply(lambda x: isinstance(x, list) and len(x) == 1)
)
launch_subset = launch_subset.loc[valid].copy()
launch_subset["DateUTC"] = pd.to_datetime(launch_subset["date_utc"], utc=True)
launch_subset = launch_subset.loc[launch_subset["DateUTC"] < CUTOFF + pd.Timedelta(days=1)].copy()
launch_subset["core_info"] = launch_subset["cores"].str[0]
launch_subset["payload_id"] = launch_subset["payloads"].str[0]
print("Launches after shape/date filters:", len(launch_subset))
launch_subset[["flight_number", "DateUTC", "rocket"]].head()

Launches after shape/date filters: 94


,flight_number,DateUTC,rocket
0,1,2006-03-24 22:30:00+00:00,5e9d0d95eda69955f709d1eb
1,2,2007-03-21 01:10:00+00:00,5e9d0d95eda69955f709d1eb
3,4,2008-09-28 23:15:00+00:00,5e9d0d95eda69955f709d1eb
4,5,2009-07-13 03:35:00+00:00,5e9d0d95eda69955f709d1eb
5,6,2010-06-04 18:45:00+00:00,5e9d0d95eda69973a809d1ec


## Enrich identifiers when the API is available

Use cached requests to avoid repeated calls for the same rocket, payload, launchpad, or core. Each output row is built as one record. **If the live enrichment endpoint fails**, load IBM's historical, enriched Falcon 9 CSV instead. A successful fallback is clearly labeled as using archived data, not as proof that the public API is online.

In [4]:
@lru_cache(maxsize=None)
def get_resource(kind: str, resource_id):
    if not resource_id or pd.isna(resource_id):
        return {}
    url = f"{SPACE_X_API}/{kind}/{resource_id}"
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        raise RuntimeError(f"SpaceX enrichment unavailable at {url}: {exc}") from exc


def build_launch_row(rec):
    core = rec["core_info"]
    rocket = get_resource("rockets", rec["rocket"])
    payload = get_resource("payloads", rec["payload_id"])
    site = get_resource("launchpads", rec["launchpad"])
    booster = get_resource("cores", core.get("core")) if core.get("core") else {}

    return {
        "FlightNumber": rec["flight_number"],
        "Date": rec["DateUTC"].strftime("%Y-%m-%d"),
        "BoosterVersion": rocket.get("name"),
        "PayloadMass": payload.get("mass_kg"),
        "Orbit": payload.get("orbit"),
        "LaunchSite": site.get("name"),
        "Outcome": f"{core.get('landing_success')} {core.get('landing_type')}",
        "Flights": core.get("flight"),
        "GridFins": core.get("gridfins"),
        "Reused": core.get("reused"),
        "Legs": core.get("legs"),
        "LandingPad": core.get("landpad"),
        "Block": booster.get("block"),
        "ReusedCount": booster.get("reuse_count"),
        "Serial": booster.get("serial"),
        "Longitude": site.get("longitude"),
        "Latitude": site.get("latitude"),
    }


try:
    records = [build_launch_row(rec) for _, rec in launch_subset.iterrows()]
    df = pd.DataFrame.from_records(records)
    source_mode = "api"
    source_description = "IBM historical launch JSON + live SpaceX v4 resource enrichment"
except RuntimeError as exc:
    print("Live SpaceX API enrichment failed:", exc)
    print("Falling back to the IBM Skills Network historical Falcon 9 dataset.")
    archived_response = session.get(ARCHIVED_FALCON9_URL, timeout=35)
    archived_response.raise_for_status()
    df = pd.read_csv(BytesIO(archived_response.content))
    source_mode = "ibm_archive"
    source_description = "IBM Skills Network archived dataset_part_1.csv (fallback)"

print("Data source:", source_description)
print("Records loaded:", len(df))
df.head()

Live SpaceX API enrichment failed: SpaceX enrichment unavailable at https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb: 525 Server Error: <none> for url: https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb
Falling back to the IBM Skills Network historical Falcon 9 dataset.
Data source: IBM Skills Network archived dataset_part_1.csv (fallback)
Records loaded: 90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## Filter Falcon 9 and handle missing payload mass

For live-enriched records, select Falcon 9 launches before imputation. When using the course's archived Falcon 9 table, keep its historical observations and columns. In both cases, convert payload mass to numeric and fill missing values *within the selected table*, avoiding chained assignments.

In [5]:
expected_columns = {
    "FlightNumber", "Date", "BoosterVersion", "PayloadMass", "Orbit",
    "LaunchSite", "Outcome", "Flights", "GridFins", "Reused", "Legs",
    "LandingPad", "Block", "ReusedCount", "Serial", "Longitude", "Latitude",
}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"The selected data source is missing columns: {sorted(missing_columns)}")

data_falcon9 = df.loc[df["BoosterVersion"].eq("Falcon 9")].copy().reset_index(drop=True)
if data_falcon9.empty:
    raise ValueError("No Falcon 9 launch records were found in the selected dataset.")
if source_mode == "api":
    data_falcon9["FlightNumber"] = range(1, len(data_falcon9) + 1)

data_falcon9["PayloadMass"] = pd.to_numeric(data_falcon9["PayloadMass"], errors="coerce")
print("Missing payload masses before:", data_falcon9["PayloadMass"].isna().sum())
payload_mean = data_falcon9["PayloadMass"].mean()
if pd.isna(payload_mean):
    raise ValueError("No numeric payload masses are available for mean imputation.")
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].fillna(payload_mean)
print("Falcon 9 observations:", len(data_falcon9))
print("Missing payload masses after:", data_falcon9["PayloadMass"].isna().sum())
data_falcon9.isna().sum().sort_values(ascending=False).head(10)

Missing payload masses before: 0
Falcon 9 observations: 90
Missing payload masses after: 0


LandingPad        26
FlightNumber       0
Date               0
PayloadMass        0
BoosterVersion     0
LaunchSite         0
Outcome            0
Flights            0
Orbit              0
GridFins           0
dtype: int64

## Export the historical Falcon 9 table

Save `outputs/dataset_part_1.csv` for any downstream work. **Check the printed data source:** if the API was unavailable, this is an IBM archived dataset, not a newly API-enriched export. Later notebooks can use IBM's separately frozen teaching data to ensure the course analyses stay comparable.

In [6]:
output_file = OUTPUT_DIR / "dataset_part_1.csv"
data_falcon9.to_csv(output_file, index=False)
print("Saved:", output_file.resolve())
print("Data source:", source_description)
print("Shape:", data_falcon9.shape)
data_falcon9.head()

Saved: C:\Users\alexa\OneDrive\Desktop\coursera\Applied Data Science Capstone\outputs\dataset_part_1.csv
Data source: IBM Skills Network archived dataset_part_1.csv (fallback)
Shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## What this notebook demonstrates

- Reading and normalizing an archived SpaceX launch JSON snapshot
- An ID-based enrichment workflow with cached REST requests, for when the source API works
- Graceful, **explicit** fallback to an independently hosted historical IBM dataset when the enrichment service is down
- Falcon 9 filtering, transparent missing-value imputation, and reproducible CSV export

**Interpretation:** `Outcome` is a first-stage landing-status field, not a mission-success label. IBM's frozen dataset documents the course-era observations; it should not be presented as current SpaceX launch data.

**Credit:** IBM Skills Network / Coursera Applied Data Science Capstone (project framework, historical datasets, and exercise design); rewritten and adapted for local VS Code execution with AI-assisted review.